# JavaScript Excel exports
Save this Deno notebook in a discovered folder, then reload the Excel add-in.


In [ ]:
/**
 * Adds two numbers locally in Excel.
 * @excelFunction JS_ADD
 * @execution local
 * @param {number} a First number.
 * @param {number} [b=0] Second number.
 * @returns {number} Sum of the inputs.
 * @example =Jupyter.JS_ADD(3, 5)
 * @see JS_GREET
 */
function js_add(a, b = 0) {
  return a + b;
}


In [ ]:
/**
 * Adds two numbers locally in Excel.
 * @excelFunction js_add_run_at_server
 * @execution server
 * @param {number} a First number.
 * @param {number} [b=0] Second number.
 * @returns {number} Sum of the inputs.
 * @example =Jupyter.js_add_run_at_server(3, 5)
 * @see JS_GREET
 */
function js_add_run_at_server(a, b = 0) {
  return a + b;
}

In [ ]:
/**
 * Formats a greeting on the notebook server.
 * @ribbonFunction JS_GREET
 * @label Greeting
 * @buttonText Say Hello
 * @execution server
 * @param {string} name Recipient.
 * @returns {Promise<string>} Greeting and Deno runtime version.
 * @remarks Requires a server connection.
 * @example name: Jim
 * @see JS_ADD
 */
async function js_greet(name) {
  return `Hello ${name} (Deno ${Deno.version.deno})`;
}


In [ ]:
/**
 * Creates a linked daily Gantt worksheet from an Excel range including headers.
 * @ribbonFunction CREATE_GANTT
 * @label Create Gantt
 * @buttonText Create Gantt
 * @execution local
 * @output status
 * @param {string} [rangeAddress="A1:D7"] Range including Task, Start, Finish, and Progress headers, e.g. A1:D7 or 'Project Plan'!A1:D7.
 * @returns {Promise<string>} Name of the created worksheet.
 * @remarks Start and Finish must be Excel dates; Progress must be a percentage between 0% and 100%. Supports 1-200 tasks and up to 366 calendar days. Existing task edits update through formulas; rerun after adding tasks or extending the timeline. Requires ExcelApi 1.7.
 * @sideEffects Creates and activates a new worksheet; does not modify the source range.
 */
async function createGantt(rangeAddress = "A1:D7") {
  if (!Office.context.requirements.isSetSupported("ExcelApi", "1.7")) {
    throw new Error("Create Gantt requires ExcelApi 1.7 or later.");
  }
  const address = rangeAddress.trim();
  const separator = address.lastIndexOf("!");
  const cells = address.slice(separator + 1);
  if (!/^\$?[A-Za-z]{1,3}\$?[1-9][0-9]*:\$?[A-Za-z]{1,3}\$?[1-9][0-9]*$/.test(cells)) {
    throw new Error("Enter a rectangular range including headers, such as A1:D7 or 'Project Plan'!A1:D7.");
  }
  return Excel.run(async context => {
    const sheets = context.workbook.worksheets;
    let sourceSheet;
    if (separator < 0) sourceSheet = sheets.getActiveWorksheet();
    else {
      let sheetName = address.slice(0, separator);
      if (sheetName.startsWith("'") && sheetName.endsWith("'")) sheetName = sheetName.slice(1, -1).replace(/''/g, "'");
      sourceSheet = sheets.getItem(sheetName);
    }
    sourceSheet.load("name");
    const source = sourceSheet.getRange(cells);
    source.load("rowCount,columnCount,rowIndex,columnIndex,address");
    sheets.load("items/name");
    await context.sync();
    if (source.rowCount < 2 || source.rowCount > 201) throw new Error("The range must contain a header row and 1-200 tasks.");
    if (source.columnCount < 4 || source.rowCount * source.columnCount > 5000) throw new Error("Select at least four columns and at most 5000 cells.");
    source.load("values");
    await context.sync();
    const names = source.values[0].map(value => String(value).trim().toLowerCase());
    const indexes = ["task", "start", "finish", "progress"].map(name => {
      const index = names.indexOf(name);
      if (index < 0 || names.lastIndexOf(name) !== index) throw new Error("Required unique column: " + name);
      return index;
    });
    const tasks = source.values.slice(1).map((row, index) => {
      const [task, start, finish, progress] = indexes.map(column => row[column]);
      const prefix = "Task row " + (index + 1) + ": ";
      if (typeof task !== "string" || !task.trim()) throw new Error(prefix + "Task must be nonempty text.");
      if (![start, finish].every(value => typeof value === "number" && Number.isFinite(value) && value >= 1 && value <= 2957000)) {
        throw new Error(prefix + "Start and Finish must be Excel dates, not text.");
      }
      if (Math.floor(finish) < Math.floor(start)) throw new Error(prefix + "Finish precedes Start.");
      if (typeof progress !== "number" || !Number.isFinite(progress) || progress < 0 || progress > 1) {
        throw new Error(prefix + "Progress must be 0%-100% (numeric 0-1).");
      }
      return {start: Math.floor(start), finish: Math.floor(finish)};
    });
    const first = Math.min(...tasks.map(task => task.start));
    const last = Math.max(...tasks.map(task => task.finish));
    const days = last - first + 1;
    if (days > 366) throw new Error("The timeline exceeds 366 days. Use a shorter project date range.");
    const existing = new Set(sheets.items.map(sheet => sheet.name.toLowerCase()));
    let name = "Gantt", suffix = 2;
    while (existing.has(name.toLowerCase())) name = "Gantt " + suffix++;
    let sheet;
    try {
      sheet = sheets.add(name);
      await context.sync();
      const title = sheet.getRange("A1:D1");
      title.merge();
      title.values = [["Project Gantt", "", "", ""]];
      title.format.font.size = 18;
      title.format.font.bold = true;
      sheet.getRange("A2:D2").values = [["Source", source.address, "Tasks", tasks.length]];
      const header = sheet.getRangeByIndexes(3, 0, 1, 4 + days);
      header.values = [["Task", "Start", "Finish", "Progress", ...Array.from({length: days}, (_, i) => first + i)]];
      header.format.fill.color = "#263238";
      header.format.font.color = "#FFFFFF";
      header.format.font.bold = true;
      header.format.rowHeight = 42;
      const dates = sheet.getRangeByIndexes(3, 4, 1, days);
      dates.numberFormat = [Array(days).fill("ddd\nd mmm")];
      dates.format.wrapText = true;
      dates.format.columnWidth = 42;
      function columnName(index) {
        let value = index + 1, result = "";
        while (value) { value--; result = String.fromCharCode(65 + value % 26) + result; value = Math.floor(value / 26); }
        return result;
      }
      // Quote worksheet names and use absolute references to the source cells.
      const sheetReference = "'" + sourceSheet.name.replace(/'/g, "''") + "'!";
      sheet.getRangeByIndexes(4, 0, tasks.length, 4).formulas = tasks.map((_, row) =>
        indexes.map(column => "=" + sheetReference + "$" + columnName(source.columnIndex + column) + "$" + (source.rowIndex + row + 2)));
      sheet.getRangeByIndexes(4, 1, tasks.length, 2).numberFormat = tasks.map(() => ["yyyy-mm-dd", "yyyy-mm-dd"]);
      sheet.getRangeByIndexes(4, 3, tasks.length, 1).numberFormat = tasks.map(() => ["0%"]);
      sheet.getRange("A:A").format.columnWidth = 180;
      sheet.getRange("B:C").format.columnWidth = 85;
      sheet.getRange("D:D").format.columnWidth = 65;
      const grid = sheet.getRangeByIndexes(4, 4, tasks.length, days);
      grid.format.rowHeight = 22;
      function fillRule(range, formula, color) {
        const rule = range.conditionalFormats.add(Excel.ConditionalFormatType.custom);
        rule.custom.rule.formula = formula;
        rule.custom.format.fill.color = color;
      }
      // Mutually exclusive rules keep progress, remaining work, and weekends distinct.
      const valid = 'AND(ISNUMBER($B5),ISNUMBER($C5),ISNUMBER($D5),$C5>=$B5,$D5>=0,$D5<=1)';
      const active = "AND(" + valid + ",E$4>=INT($B5),E$4<=INT($C5))";
      const completed = "E$4<INT($B5)+ROUNDUP((INT($C5)-INT($B5)+1)*$D5,0)";
      fillRule(grid, "=AND(" + active + "," + completed + ")", "#218C74");
      fillRule(grid, "=AND(" + active + ",NOT(" + completed + "))", "#6EA8D9");
      fillRule(grid, "=AND(WEEKDAY(E$4,2)>5,NOT(" + active + "))", "#ECEFF1");
      fillRule(dates, "=E$4=TODAY()", "#B45309");
      sheet.getRange("A3:D3").values = [["Completed: green", "Remaining: blue", "Today: amber", "Calendar days"]];
      sheet.getRange("A3:D3").format.font.size = 9;
      sheet.getRange("A3:D3").format.wrapText = true;
      sheet.getRange("A3:D3").format.rowHeight = 28;
      sheet.activate();
      sheet.freezePanes.freezeAt(sheet.getRange("A1:D4"));
      sheet.getRange("A1:" + columnName(3 + days) + (4 + tasks.length)).format.font.name = "Calibri";
      await context.sync();
      return "Created " + name + " from " + source.address + ". Dates and progress stay linked; rerun after adding tasks or extending the timeline.";
    } catch (error) {
      if (sheet) {
        try { sheet.delete(); await context.sync(); }
        catch (cleanupError) { throw new Error(error.message + ". Could not remove partial worksheet " + name + ": " + cleanupError.message); }
      }
      throw error;
    }
  });
}


In [ ]:
/**
 * Draws a Sankey image from Source, Target, and Value columns.
 * @ribbonFunction CREATE_SANKEY
 * @label Create Sankey
 * @buttonText Create Sankey
 * @execution local
 * @output status
 * @param {string} sourceRange Select the source range including headers.
 * @rangeInput sourceRange
 * @param {string} targetCell Select the cell at the diagram's top-left corner.
 * @cellInput targetCell
 * @returns {Promise<string>} Diagram creation status.
 * @remarks Positive values only; duplicate flows are combined. Cycles and self-links are not supported. Edit the source data and rerun to refresh. Requires ExcelApi 1.9. The result is an image, not an editable Excel chart.
 * @sideEffects Inserts or refreshes this action's diagram at the selected target, without changing cells.
 */
async function createSankey(sourceRange, targetCell) {
  return JupyterExcelCharts.createSankey(sourceRange, targetCell);
}
